all files in adventureWorks2 with column names

In [0]:
# Bronze (External Volume)
bronze_aw2_path = "/Volumes/main/default/fabmigration1_bronze/adventureworks2/"

df_aw2_bronze = spark.read.parquet(bronze_aw2_path)


In [0]:
# files = mssparkutils.fs.ls(bronze_base)

# production_files = [
#     f"{bronze_base}{f.name}"
#     for f in files
#     if f.name.startswith("Production.") and f.name.endswith(".parquet")
# ]


# print("Production.*  files found:")
# for p in production_files:
#     print(" -", p)

bronze_base = "/Volumes/main/default/fabmigration1_bronze/adventureworks2/"

def list_parquets(path):
    out = []
    for i in dbutils.fs.ls(path):
        if i.path.endswith("/"):
            out.extend(list_parquets(i.path))
        elif i.path.lower().endswith(".parquet"):
            out.append(i.path)
    return out

parquets = list_parquets(bronze_base)

production_files = [p for p in parquets if "/Production." in p or p.split("/")[-1].startswith("Production.")]
print(f"Found {len(production_files)} parquet files under Production.*")
for p in production_files[:50]:
    print(" -", p)    


In [0]:
from pyspark.sql import functions as F

def clean_df(df):
    # trim() a strings
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # Replace nulls (solo si hay columnas de ese tipo)
    fill_dict = {col: "" for col, dtype in df.dtypes if dtype == "string"}
    fill_dict_numeric = {
        col: 0
        for col, dtype in df.dtypes
        if dtype in ["int", "bigint", "double", "float"]
    }

    if fill_dict:                 # <- evita fillna({}) que truena
        df = df.fillna(fill_dict)

    if fill_dict_numeric:         # <- evita fillna({}) que truena
        df = df.fillna(fill_dict_numeric)

    return df

In [0]:
from typing import List, Optional
from pyspark.sql import DataFrame, functions as F, types as T
from pyspark.databricks.sql import functions as dbf
import re


def standardizedate_df(
    df: DataFrame,
    date_cols: Optional[List[str]] = None,
    input_formats: Optional[List[str]] = None,
    cast_dates_as: str = "date",        # "date" | "timestamp"
    ingest_date: Optional[str] = None,  # "yyyy-MM-dd" o None
    keep_raw: bool = True,              # guarda valor original
    add_parse_flag: bool = True          # flag de parse fallido
) -> DataFrame:
    """
    Estandariza columnas de fecha/timestamp de forma segura y tolerante.

    - Usa try_to_timestamp (ANSI-safe)
    - Maneja múltiples formatos
    - Convierte epoch seconds/millis
    - Valores inválidos -> NULL (no texto)
    - Opcional: conserva valor original y flag de error
    """

    # ---------------------------------------------------
    # 0) Trim de strings
    # ---------------------------------------------------
    for col_name, dtype in df.dtypes:
        if dtype == "string":
            df = df.withColumn(col_name, F.trim(F.col(col_name)))

    # ---------------------------------------------------
    # 1) Detección de columnas date-like (heurística segura)
    # ---------------------------------------------------
    if date_cols is None:
        pattern = re.compile(r"(^|_)(date|dt|ts|timestamp|fecha)(_|\b)")
        date_cols = [
            c for c, t in df.dtypes
            if t in ("string", "date", "timestamp")
            and pattern.search(c.lower())
        ]

    # ---------------------------------------------------
    # 2) Formatos soportados
    # ---------------------------------------------------
    if input_formats is None:
        input_formats = [
            # Date
            "yyyy-MM-dd", "yyyy/MM/dd",
            "dd-MM-yyyy", "dd/MM/yyyy",
            "MM-dd-yyyy", "MM/dd/yyyy",
            "yyyyMMdd", "ddMMyyyy",

            # Date + time
            "yyyy-MM-dd HH:mm:ss",
            "dd/MM/yyyy HH:mm:ss",
            "MM/dd/yyyy HH:mm:ss",

            # ISO 8601
            "yyyy-MM-dd'T'HH:mm:ss",
            "yyyy-MM-dd'T'HH:mm:ss.SSS",
            "yyyy-MM-dd'T'HH:mm:ssX",
            "yyyy-MM-dd'T'HH:mm:ssXX",
            "yyyy-MM-dd'T'HH:mm:ssXXX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSXX",
            "yyyy-MM-dd'T'HH:mm:ss.SSSXXX"
        ]

    dtype_map = dict(df.dtypes)

    # ---------------------------------------------------
    # 3) Parseo seguro por columna
    # ---------------------------------------------------
    for c in date_cols:
        t = dtype_map.get(c, "")

        if t == "string":

            # --- Preservar valor original
            if keep_raw:
                df = df.withColumn(f"{c}__raw", F.col(c))

            # --- Parse por formatos (ANSI-safe)
            parsed = None
            for fmt in input_formats:
                candidate = dbf.try_to_timestamp(F.col(c), F.lit(fmt))
                parsed = candidate if parsed is None else F.coalesce(parsed, candidate)

            # --- Epoch seconds / millis (solo si es numérico)
            digits = F.regexp_replace(F.col(c), r"\D", "")
            epoch_ts = (
                F.when(F.length(digits) == 13,
                       F.to_timestamp(F.from_unixtime(digits.cast("double") / 1000)))
                 .when(F.length(digits) == 10,
                       F.to_timestamp(F.from_unixtime(digits.cast("double"))))
            )

            parsed = F.coalesce(parsed, epoch_ts)

            # --- Asignar: si no parsea → NULL
            df = df.withColumn(c, parsed)

            # --- Flag de error de parseo
            if add_parse_flag:
                df = df.withColumn(
                    f"{c}__parse_failed",
                    F.col(c).isNull() & F.col(f"{c}__raw").isNotNull()
                )

        # ---------------------------------------------------
        # 4) Normalización final del tipo
        # ---------------------------------------------------
        if cast_dates_as == "date":
            df = df.withColumn(c, F.to_date(F.col(c)))
        else:
            df = df.withColumn(c, F.col(c).cast("timestamp"))

    # ---------------------------------------------------
    # 5) ingest_date (para particionado)
    # ---------------------------------------------------
    if ingest_date:
        df = df.withColumn("ingest_date", F.lit(ingest_date).cast(T.DateType()))
    else:
        df = df.withColumn("ingest_date", F.current_date())

    return df


In [0]:
for file_path in production_files:
    print(f"\nProcessing: {file_path}")

    # Base name, ej: Production.Culture.parquet
    #file_name = file_path.split("/")[-1].replace(".parquet", "")
    #display(file_name)

    # Base name
    file_name = file_path.split("/")[-1]             
    file_name = file_name.replace(".parquet", "") 
    file_name = file_name.replace(".", "_")   
    #file_name = "silver_" + file_name.lower()
    file_name = file_name.lower()         
    display(file_name)
    

    # Read parquet file
    df = spark.read.parquet(file_path) 
    df.printSchema()

    # Clean
    df_clean = clean_df(df)

    # dates
    df_table= standardizedate_df(df_clean)

    # Destination silver path
    out_path = f"main.silver.{file_name}"   
    print(f"Saved in Silver: {out_path}")

    # 🔑 ALINEACIÓN DE ESQUEMA
    if spark.catalog.tableExists(out_path):
        target_schema = spark.table(out_path).schema
        df_table = df_table.select(
            *[F.col(f.name).cast(f.dataType).alias(f.name) for f in target_schema]
        )

    df_table.write \
        .format("delta") \
        .mode("overwrite") \
        .saveAsTable(out_path)

    # Save  (Silver)  
    #df_table.write.format("delta").mode("overwrite").option("overwriteSchema", "true").save(out_path)    

    # Save as Delta table (Silver)
    # df_table.write \
    #     .format("delta") \
    #     .mode("overwrite") \
    #     .saveAsTable(out_path)

    # #Next lets register the delta table to use in Synapse
    # # Create schema if not exist
    # targetEnv = "silver"
    # spark.sql(f"""CREATE SCHEMA IF NOT EXISTS {targetEnv}""")
    
    # #Register table to use in Synapse
    # spark.sql(f""" CREATE TABLE IF NOT EXISTS {targetEnv}.{file_name.replace("silver_", "").lower()} USING DELTA LOCATION '{out_path}/' """)
   

In [0]:
"""
# Base path 
base_path = "abfss://fabmigation1@fabmigration.dfs.core.windows.net/silver/adventureworks/"

# Synapse util
from notebookutils import mssparkutils

dry_run = True   # ← Change to False to remove

items = mssparkutils.fs.ls(base_path)
targets = [it for it in items if it.isDir and it.name.strip("/").lower().startswith("production_")]

print("Folders founded starting with 'production_':")
for t in targets:
    print(" -", t.path)

if not dry_run:
    for t in targets:
        print("Borrando →", t.path)
        mssparkutils.fs.rm(t.path, True)   # True = recursive
    print("Done ✔")
else:
    print("\nDry-run active: nothing removed. Change dry_run=False to execute.")
   """

In [0]:
#Create external tables
#for file_path in production_files:
#    table_name = file_path.split("/")[-1].replace(".parquet", "")
#    out_path = f"{silver_base}{table_name}/"
#
#    spark.sql(f"""
#        CREATE OR REPLACE TABLE silver_{table_name}
#        USING PARQUET
#        LOCATION '{out_path}'
#    """)